# Assignment: Build a Chatbot using Hugging Face Transformers


## Objective
Build a simple conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.


## Learning Outcomes
After completing this assignment, students will be able to:
Understand how transformer-based language models work

Use pre-trained NLP models from Hugging Face Model Hub

Perform text generation using transformer models

Build an interactive conversational system

Understand prompt-based text generation


## Tools and Technologies

Students should use the following:

Python

Hugging Face Transformers

TensorFlow or PyTorch

Jupyter Notebook / VS Code


## Task Description
You are required to build a console-based chatbot that communicates with users in natural language using a pre-trained transformer model. The chatbot should dynamically generate responses and simulate a basic conversational AI assistant.


## Expected Pipeline Flow
User Input → Model Processing → Response Generation → Display Output → Loop Until Exit


## Task Breakdown
## 1. Model Loading (Mandatory)

Load a pre-trained transformer model from Hugging Face

Example models: GPT-2, DialoGPT


In [1]:
!pip install transformers torch
!pip install ipywidgets

## 2. User Input Handling
Accept user input from the console

Ensure smooth interaction


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"  # suppresses the warning
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"  # suppresses advisory warnings

print("Libraries imported successfully!")

Libraries imported successfully!


## 3. Response Generation
Generate responses using the transformer model

Use prompt-based text generation


In [3]:
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer for '{MODEL_NAME}'...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model '{MODEL_NAME}'...")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("\nModel and tokenizer loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Loading tokenizer for 'microsoft/DialoGPT-medium'...
Loading model 'microsoft/DialoGPT-medium'...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


Model and tokenizer loaded successfully!
Model parameters: 354,823,168


## 4. Continuous Conversation
Maintain conversation flow

Allow multiple interactions


## 5. Exit Condition
Stop chatbot when user types:

exit or quit


In [ ]:
def generate_response(user_input, chat_history_ids=None):
    
    user_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"   # return as PyTorch tensor
    )

    bot_input_ids = (
        torch.cat([chat_history_ids, user_input_ids], dim=-1)
        if chat_history_ids is not None
        else user_input_ids
    )

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    new_chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,   # explicitly passed to avoid warning
        max_length=1000,          # max total tokens (history + response)
        do_sample=True,           # use sampling instead of greedy decoding
        top_k=50,                 # limit sampling to top-50 tokens
        top_p=0.95,               # nucleus sampling threshold
        temperature=0.75,         # creativity control
        pad_token_id=tokenizer.eos_token_id  # avoid padding warnings
    )

    response_text = tokenizer.decode(
        new_chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True  
    )

    return response_text, new_chat_history_ids

def run_chatbot():
    """
    Main chatbot loop.
    Continuously accepts user input, generates responses using DialoGPT,
    and maintains conversation history until the user types 'exit' or 'quit'.
    """

    print("=" * 60)
    print("       AI CHATBOT — Powered by DialoGPT (Hugging Face)")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("         (Type 'exit' or 'quit' to end the conversation)")
    print("-" * 60)

    # Initialize conversation history as None (no history on first turn)
    chat_history_ids = None

    # Main conversation loop
    while True:

        # Accept user input from the console
        user_input = input("You: ").strip()

        # Skip empty inputs gracefully
        if not user_input:
            print("Chatbot: Please type something so I can help you!")
            continue

        # Check for exit condition — stop chatbot if user types 'exit' or 'quit'
        if user_input.lower() in ["exit", "quit"]:
            print("-" * 60)
            print("Chatbot: Goodbye! It was a pleasure talking with you. Have a great day!")
            print("=" * 60)
            break  # Exit the loop and end the chatbot session

        # Generate a response using the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Display the chatbot's response
        print(f"Chatbot: {response}")
        print("-" * 60)

print("Response generation function defined successfully!")
# Start the chatbot
run_chatbot()


Response generation function defined successfully!
       AI CHATBOT — Powered by DialoGPT (Hugging Face)
Chatbot: Hello! I am your AI assistant. How can I help you today?
         (Type 'exit' or 'quit' to end the conversation)
------------------------------------------------------------
